In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import math

# Start Spark
spark = SparkSession.builder.appName("UserCF").getOrCreate()

# Dataset
data = [
    ("A", 5, 1, 5, 1, 2),
    ("B", 1, 4, 2, 5, 1),
    ("C", 4, 2, 4, 2, 1),
    ("D", 2, 5, 1, 4, 5),
    ("E", 5, None, 4, None, 1)
]

columns = ["User", "Action", "Comedy", "SciFi", "Romance", "Horror"]
df = spark.createDataFrame(data, columns)

df.show()

+----+------+------+-----+-------+------+
|User|Action|Comedy|SciFi|Romance|Horror|
+----+------+------+-----+-------+------+
|   A|     5|     1|    5|      1|     2|
|   B|     1|     4|    2|      5|     1|
|   C|     4|     2|    4|      2|     1|
|   D|     2|     5|    1|      4|     5|
|   E|     5|  NULL|    4|   NULL|     1|
+----+------+------+-----+-------+------+



In [3]:
users = df.collect()

ratings = {}
for row in users:
    ratings[row["User"]] = {
        "Action": row["Action"],
        "Comedy": row["Comedy"],
        "SciFi": row["SciFi"],
        "Romance": row["Romance"],
        "Horror": row["Horror"]
    }

In [4]:
def cosine_similarity(u1, u2):
    common = []
    for key in u1:
        if u1[key] is not None and u2[key] is not None:
            common.append(key)

    num = sum(u1[k]*u2[k] for k in common)
    den1 = math.sqrt(sum(u1[k]**2 for k in common))
    den2 = math.sqrt(sum(u2[k]**2 for k in common))

    if den1 == 0 or den2 == 0:
        return 0
    return num / (den1 * den2)

In [5]:
target = ratings["E"]

similarities = {}
for user in ratings:
    if user != "E":
        similarities[user] = cosine_similarity(target, ratings[user])

print("Similarities:", similarities)

Similarities: {'A': 0.9869072350796488, 'B': 0.8819171036881969, 'C': 0.993848322297969, 'D': 0.5352643613280604}


In [6]:
def predict(item):
    num = 0
    den = 0
    for user in ratings:
        if user != "E" and ratings[user][item] is not None:
            sim = similarities[user]
            num += sim * ratings[user][item]
            den += sim
    return num / den if den != 0 else 0

comedy_pred = predict("Comedy")
romance_pred = predict("Romance")

print("Predicted Comedy:", round(comedy_pred))
print("Predicted Romance:", round(romance_pred))

Predicted Comedy: 3
Predicted Romance: 3
